In [8]:
class PlayerProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.40
        else:
            return 0.45

    def is_forward(self): return self.position == "forward"
    def is_midfielder(self): return self.position == "midfielder"
    def is_defender(self): return self.position == "defender"

    def input_match_stats(self, minutes=0, goals=0, assists=0,
                          yellow_card=False, red_card=False, clean_sheet=False, goals_conceded=0,
                          total_shots=0, shots_on_target=0, shots_off_target=0, tackles_loss=0,
                          total_passes=0, accurate_passes=0,
                          expected_goals=0, expected_assists=0,
                          big_chances_missed=0,
                          successful_dribbles=0, total_dribbles=0, conceded_penalty=0, missed_penalty=0,
                          accurate_crosses=0, total_crosses=0,
                          accurate_long_balls=0, total_long_balls=0,
                          dispossessed=0, tackles_won=0, interceptions=0,
                          clearances=0, ball_recoveries=0,
                          dribbled_past=0, duels_won=0, duels_lost=0,
                          ground_duels_won=0, ground_duels_total=0,
                          aerial_duels_won=0, aerial_duels_total=0, own_goal=0,
                          fouled=0, number_of_fouls=0):

        boost = self.get_rating_boost()

        # Goals and Assists
        self.adjust_rating(goals * 1)
        self.adjust_rating(assists * 1)

        if goals_conceded > 2:
            self.adjust_rating(-1)
            

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)
                #Clean Sheet Bonus
        if clean_sheet:
            self.adjust_rating(boost)

        # Shot Accuracy
        if total_shots > 0:
            accuracy = shots_on_target / total_shots
            if accuracy >= 0.7:
                self.adjust_rating(boost)
            shots_off_target = total_shots - shots_on_target
            off_target_ratio = shots_off_target / total_shots
            if off_target_ratio >= 0.75:
                self.adjust_rating(-0.3)
            elif off_target_ratio >= 0.5:
                self.adjust_rating(-0.2)
            elif off_target_ratio >= 0.3:
                self.adjust_rating(-0.1)

        # xG comparison
        if goals > expected_goals:
            self.adjust_rating(boost)
        elif expected_goals > goals:
            self.adjust_rating(-0.2)

        # xA comparison
        if assists > expected_assists:
            self.adjust_rating(boost)
        elif expected_assists > assists:
            self.adjust_rating(-0.2)

        # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Dribbling Success Rate
        if total_dribbles > 0:
            dribble_rate = successful_dribbles / total_dribbles
            if dribble_rate >= 0.8:
                self.adjust_rating(boost)
            elif dribble_rate >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.2)
            if tackles_won > tackles_loss:
                self.adjust_rating(boost)
            elif tackles_won < tackles_loss:
                self.adjust_rating(-boost)
             
        # Crossing
        if total_crosses > 0:
            crossing_rate = accurate_crosses / total_crosses
            if crossing_rate >= 0.5:
                self.adjust_rating(boost)
            elif crossing_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Long Balls
        if total_long_balls > 0:
            long_ball_rate = accurate_long_balls / total_long_balls
            if long_ball_rate >= 0.5:
                self.adjust_rating(boost)
            elif long_ball_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Possession Loss
        if dispossessed >= 3:
            self.adjust_rating(-0.1 * dispossessed)

        # Defensive Metrics
        if tackles_won >= 2:
            self.adjust_rating(tackles_won * 0.1)
        if interceptions >= 2:
            self.adjust_rating(interceptions * 0.1)
        if clearances >= 2:
            self.adjust_rating(clearances * 0.05)
        if ball_recoveries >= 5:
            self.adjust_rating(ball_recoveries * 0.05)
        if dribbled_past >= 2:
            self.adjust_rating(-0.1 * dribbled_past)

        # Duels
        total_duels = duels_won + duels_lost
        if total_duels > 0:
            duel_win_rate = duels_won / total_duels
            if duel_win_rate >= 0.6:
                self.adjust_rating(boost)
            elif duel_win_rate < 0.4:
                self.adjust_rating(-boost)

        if ground_duels_total > 0:
            ground_rate = ground_duels_won / ground_duels_total
            if ground_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        if aerial_duels_total > 0:
            aerial_rate = aerial_duels_won / aerial_duels_total
            if aerial_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        # Fouls and Being Fouled
        self.adjust_rating(fouled * 0.05)
        self.adjust_rating(-number_of_fouls * 0.1)

        # Played Full 90 Minutes
        if minutes >= 90:
            self.adjust_rating(0.2)

        # Big Chances Missed
        if big_chances_missed > 0:
            self.adjust_rating(-0.3 * big_chances_missed)

        if conceded_penalty > 0:
            self.adjust_rating(-2)
        if missed_penalty > 0:
            self.adjust_rating(-2)
        if minutes >= 90:
            self.adjust_rating(0.2) 
        if own_goal > 0:
            self.adjust_rating(-2)

class GoalkeeperProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.4
        else:
            return 0.5

    def input_match_stats(self,
                          saves=0,
                          goals_conceded=0,
                          xG_faced=0,
                          saves_inside_box=0,
                          saves_outside_box=0,
                          touches=0,
                          goals_prevented=0,
                          errors=0,
                          minutes_played=0,
                          penalty_saves=0,
                          total_passes=0,
                          accurate_passes=0,
                          total_long_balls=0,
                          accurate_long_balls=0,
                          yellow_card=False,
                          red_card=False,
                          clean_sheet=False):

        boost = self.get_rating_boost()

        # Saves
        self.adjust_rating(saves * 0.1)
        if saves_inside_box > 0:
            self.adjust_rating(saves_inside_box * 0.15)
        if saves_outside_box > 0:
            self.adjust_rating(saves_outside_box * 0.1)

        # Goals Conceded vs xG
        if goals_conceded < xG_faced:
            self.adjust_rating(boost)
        elif goals_conceded > xG_faced:
            self.adjust_rating(-boost)

                # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Goals prevented
        if goals_prevented > 0:
            self.adjust_rating(goals_prevented * 0.2)

        # Errors
        if errors == 0:
            self.adjust_rating(boost * 0.5)
        elif errors > 0:
            self.adjust_rating(-0.5 * errors)

        # Touches which can lead to distribution
        if touches >= 40:
            self.adjust_rating(boost * 0.5)
        elif touches >= 25:
            self.adjust_rating(boost * 0.2)

        # Penalty saves
        if penalty_saves > 0:
            self.adjust_rating(penalty_saves * 0.8)

        # Clean sheet 
        if clean_sheet and minutes_played >= 70:
            self.adjust_rating(0.5)

        # Played full match
        if minutes_played >= 90:
            self.adjust_rating(0.2)

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)


In [28]:
#Home Team
brazil_world_cup_starting_xi_vs_morocco = [
    ["Alisson", "GK", "Liverpool", 86,],
    ["Douglas Santos", "RB", "Zenit", 76,],
    ["Ibanez", "LB", "Al-Ahli", 79,],
    ["Gabriel", "CB", "Arsenal", 88,],
    ["Marquinhos", "CB", "PSG", 86,],
    ["Casemiro", "CDM", "Manchester United", 83,],
    ["Bruno Guimaraes", "CM", "Newcastle", 86,],
    ["Vinicius Jr", "LW", "Real Madrid", 88,],
    ["Lucas Paqueta", "CM", "Flamengo", 80,],
    ["Raphinha", "RW", "Barcelona", 87,],
    ["Igor Thiago", "ST", "Brentford", 82,]

]
brazil_world_cup_bench_vs_morocco= [
    ["Fabinho", "CDM", "Al-Itihad", 78,],
    ["Matheus Cunha", "CAM", "Manchester United", 84,],
    ["Luis Henrique", "RW", "Zenit", 80,],
    ["Danilo", "RB", "Flamengo", 79,],
    ["Danilo Santos", "CM", "Botafogo", 80,],

]


In [30]:
gk = GoalkeeperProfile("Alisson", "Brazil", "goalkeeper", 86)
gk.input_match_stats(
    minutes_played=90,
    saves=2,
    saves_inside_box=1,
    saves_outside_box=1,
    goals_conceded=1,
    xG_faced=0.74,
    goals_prevented=-0.26,
    total_passes=27,
    accurate_passes=22,
    total_long_balls=8,
    accurate_long_balls=3,
    touches=33,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Alisson's match rating: 6.76


In [32]:
player = PlayerProfile("Ibanez", "Brazil", "defender", 79)

player.input_match_stats(
    minutes=45,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=16,
    total_passes=17,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=2,
    duels_won=4,
    duels_lost=5,
    ground_duels_won=4,
    ground_duels_total=9,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=2,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ibanez's match rating: 5.72


In [34]:
player = PlayerProfile("Marquinhos", "Brazil", "defender", 86)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=75,
    total_passes=79,
    expected_goals=0.09,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=4,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=3,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=3,
    aerial_duels_total=3,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Marquinhos's match rating: 6.53


In [36]:
player = PlayerProfile("Gabriel", "Brazil", "defender", 88)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=82,
    total_passes=85,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=6,
    interceptions=0,
    ball_recoveries=5,
    dribbled_past=0,
    duels_won=3,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=2,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Gabriel's match rating: 7.20


In [38]:
player = PlayerProfile("Santos", "Brazil", "defender", 76)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=53,
    total_passes=62,
    expected_goals=0,
    expected_assists=0.03,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=2,
    total_long_balls=5,
    dispossessed=0,
    tackles_won=7,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=6,
    dribbled_past=1,
    duels_won=7,
    duels_lost=2,
    ground_duels_won=7,
    ground_duels_total=9,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Santos's match rating: 8.60


In [40]:
player = PlayerProfile("Paqueta", "Brazil", "midfielder", 80)

player.input_match_stats(
    minutes=61,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=31,
    total_passes=39,
    expected_goals=0.18,
    expected_assists=0.03,
    successful_dribbles=2,
    total_dribbles=4,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=4,
    tackles_loss=0,
    clearances=3,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=2,
    duels_won=10,
    duels_lost=8,
    ground_duels_won=9,
    ground_duels_total=16,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=3,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Paqueta's match rating: 7.04


In [42]:
player = PlayerProfile("Guimaraes", "Brazil", "midfielder", 86)

player.input_match_stats(
    minutes=80,
    goals=0,
    assists=1,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=34,
    total_passes=38,
    expected_goals=0.01,
    expected_assists=0.02,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=3,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=2,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=2,
    duels_won=6,
    duels_lost=7,
    ground_duels_won=5,
    ground_duels_total=11,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Guimaraes's match rating: 7.56


In [44]:
player = PlayerProfile("Casemiro", "Brazil", "midfielder", 83)

player.input_match_stats(
    minutes=45,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=18,
    expected_goals=0,
    expected_assists=0.05,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=3,
    interceptions=1,
    ball_recoveries=5,
    dribbled_past=3,
    duels_won=3,
    duels_lost=6,
    ground_duels_won=2,
    ground_duels_total=8,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=1,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Casemiro's match rating: 5.30


In [46]:
player = PlayerProfile("Raphinha", "Brazil", "midfielder", 87)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=20,
    total_passes=31,
    expected_goals=0.15,
    expected_assists=0.33,
    successful_dribbles=1,
    total_dribbles=6,
    accurate_crosses=2,
    total_crosses=4,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=3,
    duels_won=4,
    duels_lost=8,
    ground_duels_won=3,
    ground_duels_total=11,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Raphinha's match rating: 6.00


In [48]:
player = PlayerProfile("Igor Thiago", "Brazil", "forward", 82)

player.input_match_stats(
    minutes=62,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=5,
    total_passes=6,
    expected_goals=0.66,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=3,
    duels_lost=4,
    ground_duels_won=2,
    ground_duels_total=6,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Igor Thiago's match rating: 5.87


In [50]:
player = PlayerProfile("Vinicius Jr", "Brazil", "forward", 88)

player.input_match_stats(
    minutes=90,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=26,
    total_passes=31,
    expected_goals=0.08,
    expected_assists=0.09,
    successful_dribbles=0,
    total_dribbles=8,
    accurate_crosses=0,
    total_crosses=3,
    accurate_long_balls=4,
    total_long_balls=5,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=0,
    duels_won=3,
    duels_lost=12,
    ground_duels_won=3,
    ground_duels_total=14,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=2,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Vinicius Jr's match rating: 7.55


In [52]:
player = PlayerProfile("Danilo", "Brazil", "defender", 79)

player.input_match_stats(
    minutes=45,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=32,
    total_passes=38,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=2,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Danilo's match rating: 5.61


In [54]:
player = PlayerProfile("Fabinho", "Brazil", "midfielder", 78)

player.input_match_stats(
    minutes=45,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=15,
    total_passes=17,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=2,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=1,
    duels_lost=3,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Fabinho's match rating: 5.71


In [56]:
player = PlayerProfile("Luis Henrique", "Brazil", "forward", 80)

player.input_match_stats(
    minutes=28,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=12,
    total_passes=13,
    expected_goals=0,
    expected_assists=0.04,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=2,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Luis Henrique's match rating: 7.07


In [58]:
player = PlayerProfile("Cunha", "Brazil", "forward", 84)

player.input_match_stats(
    minutes=29,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=5,
    total_passes=9,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=1,
    duels_won=3,
    duels_lost=3,
    ground_duels_won=3,
    ground_duels_total=6,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Cunha's match rating: 6.47


In [60]:
player = PlayerProfile("Danilo", "Brazil", "midfielder", 80)

player.input_match_stats(
    minutes=10,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=4,
    total_passes=4,
    expected_goals=0.09,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Danilo's match rating: 5.95


In [76]:
#AwayTeam
morocco_world_cup_starting_xi_vs_brazil = [
    ["Yassine Bounou", "GK", "Al Hilal", 82,],
    ["Achraf Hakimi", "RB", "Paris Saint-Germain", 89,],
    ["Chadi Riad", "CB", "Crystal Palace", 76,],
    ["Issa Diop", "CB", "Fulham", 75,],
    ["Noussair Mazraoui", "LB", "Manchester United", 82,],
    ["Ayyoub Bouaddi", "CM", "Lille", 79,],
    ["Neil El Aynaoui", "CM", "Roma", 77,],
    ["Azzedine Ounahi", "CAM", "Girona", 78,],
    ["Brahim Diaz", "RM", "Real Madrid", 79,],
    ["Bilal El Khannouss", "LM", "Stuttgart", 78,],
    ["Ismael Saibari", "CF", "PSV Eindhoven", 80,],
]

morocco_world_cup_bench_vs_brazil = [
    ["Ayoub Amaimouni-Echghouyabe", "ST", "Eintracht Frankfurt", 72, None, None],
    ["Chemsdine Talbi", "LM", "Sunderland", 75, None, None],
    ["Soufiane Rahimi", "ST", "Al Ain", 76, None, None],
    ["Anass Salah-Eddine", "LB", "PSV Eindhoven", 74, None, None],

]   

In [78]:
gk = GoalkeeperProfile("Bounou", "Morocco", "goalkeeper", 82)
gk.input_match_stats(
    minutes_played=90,
    saves=4,
    saves_inside_box=1,
    saves_outside_box=3,
    goals_conceded=1,
    xG_faced=1.38,
    goals_prevented=0.38,
    total_passes=35,
    accurate_passes=17,
    total_long_balls=18,
    accurate_long_balls=0,
    touches=45,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Bounou's match rating: 7.43


In [80]:
player = PlayerProfile("Mazraoui", "Morocco", "defender", 82)

player.input_match_stats(
    minutes=80,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=28,
    total_passes=32,
    expected_goals=0,
    expected_assists=0.05,
    successful_dribbles=2,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=5,
    tackles_loss=0,
    clearances=5,
    interceptions=5,
    ball_recoveries=5,
    dribbled_past=2,
    duels_won=7,
    duels_lost=6,
    ground_duels_won=7,
    ground_duels_total=12,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Mazraoui's match rating: 7.87


In [82]:
player = PlayerProfile("Riad", "Morocco", "defender", 76)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=34,
    total_passes=40,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=2,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=3,
    interceptions=2,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=2,
    duels_lost=0,
    ground_duels_won=2,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Riad's match rating: 8.05


In [84]:
player = PlayerProfile("Diop", "Morocco", "defender", 75)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=42,
    total_passes=53,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=3,
    total_long_balls=11,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Diop's match rating: 7.10


In [86]:
player = PlayerProfile("Hakimi", "Morocco", "defender", 89)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=3,
    shots_on_target=0,
    accurate_passes=44,
    total_passes=47,
    expected_goals=0.18,
    expected_assists=0.11,
    successful_dribbles=0,
    total_dribbles=3,
    accurate_crosses=2,
    total_crosses=4,
    accurate_long_balls=5,
    total_long_balls=5,
    dispossessed=2,
    tackles_won=6,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=5,
    dribbled_past=2,
    duels_won=11,
    duels_lost=12,
    ground_duels_won=11,
    ground_duels_total=20,
    aerial_duels_won=0,
    aerial_duels_total=3,
    fouled=5,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hakimi's match rating: 6.90


In [88]:
player = PlayerProfile("Bouaddi", "Morocco", "midfielder", 79)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=60,
    total_passes=66,
    expected_goals=0,
    expected_assists=0.03,
    successful_dribbles=3,
    total_dribbles=5,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=2,
    tackles_won=4,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=6,
    dribbled_past=0,
    duels_won=9,
    duels_lost=6,
    ground_duels_won=9,
    ground_duels_total=14,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Bouaddi's match rating: 8.65


In [90]:
player = PlayerProfile("El Aynaoui", "Morocco", "midfielder", 77)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=0,
    accurate_passes=47,
    total_passes=53,
    expected_goals=0.22,
    expected_assists=0.02,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=4,
    tackles_loss=0,
    clearances=2,
    interceptions=1,
    ball_recoveries=4,
    dribbled_past=0,
    duels_won=9,
    duels_lost=8,
    ground_duels_won=8,
    ground_duels_total=13,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=3,
    number_of_fouls=4,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

El Aynaoui's match rating: 7.25


In [92]:
player = PlayerProfile("Diaz", "Morocco", "midfielder", 79)

player.input_match_stats(
    minutes=65,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=19,
    total_passes=19,
    expected_goals=0.2,
    expected_assists=0.1,
    successful_dribbles=1,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=3,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=1,
    duels_won=4,
    duels_lost=6,
    ground_duels_won=4,
    ground_duels_total=10,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=3,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Diaz's match rating: 7.50


In [94]:
player = PlayerProfile("El Khannouss", "Morocco", "midfielder", 78)

player.input_match_stats(
    minutes=80,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=24,
    total_passes=28,
    expected_goals=0.02,
    expected_assists=0.03,
    successful_dribbles=2,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=2,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=4,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=6,
    dribbled_past=0,
    duels_won=7,
    duels_lost=3,
    ground_duels_won=7,
    ground_duels_total=9,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

El Khannouss's match rating: 7.99


In [98]:
player = PlayerProfile("Ounahi", "Morocco", "midfeilder", 78)

player.input_match_stats(
    minutes=65,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=33,
    total_passes=36,
    expected_goals=0,
    expected_assists=0.03,
    successful_dribbles=4,
    total_dribbles=8,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=7,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ounahi's match rating: 7.02


In [102]:
player = PlayerProfile("Saibari", "Morocco", "forard", 80)

player.input_match_stats(
    minutes=89,
    goals=1,
    assists=0,
    total_shots=3,
    shots_on_target=1,
    accurate_passes=22,
    total_passes=24,
    expected_goals=0.71,
    expected_assists=0.03,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=4,
    duels_lost=4,
    ground_duels_won=3,
    ground_duels_total=5,
    aerial_duels_won=1,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Saibari's match rating: 8.38
